[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/04_polynomial_and_spline_interpolation/first_principles.ipynb)

# Topic 04: Polynomial and Spline Interpolation

## 1. First-Principles Intuition & Motivation

A function is almost never handed to us as a formula. It arrives as a **table**: sensor readings at sampling instants, a physical constant tabulated at a few temperatures, an expensive simulation evaluated at a handful of design points, a learning-rate value specified at a few milestones. The question that immediately follows is unavoidable:

> Given $(x_0, y_0), \ldots, (x_n, y_n)$ with distinct nodes $x_i$, what is the value *between* the samples?

Interpolation answers this by choosing a function $p$ from some agreed family, subject to $p(x_i) = y_i$ for every $i$. Polynomials are the natural first family: they are closed under addition and multiplication, differentiate and integrate in closed form, are evaluated with only $+$ and $\times$ (so hardware loves them), and by Weierstrass's theorem they can approximate *any* continuous function on a closed interval arbitrarily well.

That last fact is seductive and misleading. Weierstrass guarantees that a *good* polynomial exists; it says nothing about the polynomial that happens to pass through your particular nodes. The gap between "a good approximant exists" and "interpolation at these points finds it" is precisely where the subject lives.

### The two questions of interpolation

**Question 1 — Is the interpolant well defined?** Yes, and uniquely: through $n+1$ distinct nodes there is exactly one polynomial of degree $\le n$. That single object has many *representations* — monomial coefficients, Lagrange basis, Newton divided differences, barycentric weights — which are algebraically identical but computationally very different. Choosing a representation is choosing an algorithm, not a mathematical object.

**Question 2 — How large is the error between nodes?** The answer is the error theorem

$$
f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!}\,\omega_{n+1}(x), \qquad \omega_{n+1}(x) = \prod_{i=0}^{n}(x - x_i),
$$

which factors the error into something we cannot control (the size of $f^{(n+1)}$) and something we *entirely* control (the node polynomial $\omega_{n+1}$, which depends only on where we sample). All of interpolation theory is the study of $\omega_{n+1}$.

For equispaced nodes $\omega_{n+1}$ is small in the middle of the interval and enormous near the ends — its peak near the endpoints exceeds its central peak by a factor growing like $2^{n}$. Combined with a function whose high derivatives grow (any function with a complex-plane singularity near the interval, such as $1/(1 + 25x^2)$ with poles at $\pm i/5$), this produces the **Runge phenomenon**: adding equispaced points makes the interpolant *worse*, diverging without bound. The fix is to move the nodes so that $\omega_{n+1}$ is as flat as possible — that is exactly the Chebyshev distribution $x_k = \cos(k\pi/n)$ — or to abandon high degree and stitch low-degree pieces together into a **spline**.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Interpolation problem).** Given distinct nodes $x_0, \ldots, x_n \in [a,b]$ and values $y_0, \ldots, y_n$, find $p \in \mathbb{P}_n$ (polynomials of degree at most $n$) with $p(x_i) = y_i$ for $i = 0, \ldots, n$. When $y_i = f(x_i)$ we write $p_n = \Pi_n f$ and call $\Pi_n$ the interpolation operator.

**Definition 2 (Lagrange basis).** For distinct nodes,

$$
L_i(x) = \prod_{\substack{j=0 \\ j \neq i}}^{n} \frac{x - x_j}{x_i - x_j}, \qquad L_i(x_j) = \delta_{ij},
$$

so that $p_n(x) = \sum_{i=0}^{n} y_i L_i(x)$. The $\{L_i\}$ form a basis of $\mathbb{P}_n$ (the *cardinal* basis for these nodes).

**Definition 3 (Divided differences).** Recursively,

$$
f[x_i] = f(x_i), \qquad f[x_i, \ldots, x_{i+k}] = \frac{f[x_{i+1}, \ldots, x_{i+k}] - f[x_i, \ldots, x_{i+k-1}]}{x_{i+k} - x_i}.
$$

The **Newton form** of the interpolant is

$$
p_n(x) = \sum_{k=0}^{n} f[x_0, \ldots, x_k] \prod_{j=0}^{k-1}(x - x_j).
$$

**Definition 4 (Chebyshev points).** The Chebyshev polynomial of the first kind is $T_n(x) = \cos(n \arccos x)$ on $[-1,1]$. Its extreme points ("Chebyshev points of the second kind") are

$$
x_k = \cos\!\left(\frac{k\pi}{n}\right), \quad k = 0, \ldots, n,
$$

and its roots ("Chebyshev points of the first kind") are $x_k = \cos\!\left(\frac{(2k+1)\pi}{2n+2}\right)$. Both cluster quadratically at the interval endpoints with density $\propto (1 - x^2)^{-1/2}$.

**Definition 5 (Spline).** Given a knot sequence $a = t_0 \lt t_1 \lt \cdots \lt t_m = b$, a function $s$ is a **spline of degree $k$** if $s$ restricted to each $[t_j, t_{j+1}]$ is a polynomial of degree $\le k$ and $s \in C^{k-1}[a,b]$. A **cubic spline** ($k = 3$) is piecewise cubic and twice continuously differentiable.

**Definition 6 (End conditions).** For a cubic spline through $m+1$ knots there are $4m$ coefficients and $4m - 2$ interpolation/smoothness conditions, so two extra conditions are needed:
- **Natural**: $s''(a) = s''(b) = 0$.
- **Clamped (complete)**: $s'(a) = f'(a)$, $s'(b) = f'(b)$.
- **Not-a-knot**: $s'''$ is continuous across $t_1$ and $t_{m-1}$ (so the first two and last two pieces are single cubics).

**Definition 7 (Lebesgue constant).** $\Lambda_n = \max_{x \in [a,b]} \sum_{i=0}^{n} \lvert L_i(x) \rvert$; it is the operator norm $\lVert \Pi_n \rVert_\infty$ and governs both stability and near-optimality of interpolation.

**Theorem 1 (Existence and uniqueness).** For distinct nodes $x_0, \ldots, x_n$ and any data $y_0, \ldots, y_n$ there exists exactly one $p \in \mathbb{P}_n$ with $p(x_i) = y_i$. Equivalently, the Vandermonde matrix $V_{ij} = x_i^{\,j}$ is nonsingular, with

$$
\det V = \prod_{0 \le i \lt j \le n} (x_j - x_i) \neq 0 .
$$

**Theorem 2 (Interpolation error / Cauchy remainder).** Let $f \in C^{n+1}[a,b]$ and let $p_n$ interpolate $f$ at distinct nodes $x_0,\ldots,x_n \in [a,b]$. Then for every $x \in [a,b]$ there is $\xi_x \in (\min, \max)$ of the nodes and $x$ with

$$
f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!} \prod_{i=0}^{n}(x - x_i).
$$

**Theorem 3 (Divided differences and derivatives).** The leading coefficient of $p_n$ is $f[x_0, \ldots, x_n]$, and for $f \in C^{n}$ there exists $\xi$ in the span of the nodes with

$$
f[x_0, \ldots, x_n] = \frac{f^{(n)}(\xi)}{n!} .
$$

Consequently $f(x) - p_n(x) = f[x_0, \ldots, x_n, x]\,\omega_{n+1}(x)$ exactly, for any $f$ (no smoothness needed).

**Theorem 4 (Chebyshev minimax property).** Among all **monic** polynomials of degree $n$ on $[-1,1]$, the scaled Chebyshev polynomial $\tilde T_n = 2^{1-n} T_n$ uniquely minimizes the sup norm, with

$$
\min_{\text{monic } q \in \mathbb{P}_n} \max_{x \in [-1,1]} \lvert q(x) \rvert = \max_{x \in [-1,1]} \lvert \tilde T_n(x) \rvert = 2^{\,1-n}.
$$

Hence choosing the interpolation nodes to be the roots of $T_{n+1}$ minimizes $\lVert \omega_{n+1} \rVert_\infty$, giving the error bound

$$
\lVert f - p_n \rVert_\infty \le \frac{\lVert f^{(n+1)} \rVert_\infty}{2^{n}\,(n+1)!}.
$$

**Theorem 5 (Lebesgue bound / near-best approximation).** For any nodes, $\lVert f - \Pi_n f \rVert_\infty \le (1 + \Lambda_n)\, \inf_{q \in \mathbb{P}_n} \lVert f - q \rVert_\infty$. Asymptotically $\Lambda_n^{\mathrm{equi}} \sim \frac{2^{\,n+1}}{e\,n\log n}$ (exponential) while $\Lambda_n^{\mathrm{Cheb}} \sim \frac{2}{\pi}\log n + 1$ (logarithmic).

**Theorem 6 (Hermite interpolation).** Given $f(x_i)$ and $f'(x_i)$ at $n+1$ distinct nodes there is a unique $H \in \mathbb{P}_{2n+1}$ matching both, and for $f \in C^{2n+2}$

$$
f(x) - H(x) = \frac{f^{(2n+2)}(\xi_x)}{(2n+2)!} \prod_{i=0}^{n}(x - x_i)^2 .
$$

**Theorem 7 (Cubic spline: existence, uniqueness, tridiagonal system).** For each of the natural, clamped, and not-a-knot end conditions there is a unique interpolating cubic spline. Writing $h_i = x_{i+1} - x_i$ and $M_i = s''(x_i)$, the moments satisfy, for $i = 1, \ldots, n-1$,

$$
h_{i-1} M_{i-1} + 2\,(h_{i-1} + h_i)\, M_i + h_i M_{i+1} = 6 \left( \frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}} \right),
$$

a symmetric, **strictly diagonally dominant tridiagonal** system solvable in $O(n)$ operations by the Thomas algorithm.

**Theorem 8 (Spline accuracy).** For the clamped (complete) cubic spline interpolating $f \in C^4[a,b]$ on a mesh of maximum width $h$,

$$
\lVert f - s \rVert_\infty \le \frac{5}{384}\, h^4 \lVert f^{(4)} \rVert_\infty, \qquad \lVert f' - s' \rVert_\infty \le \frac{1}{24} h^3 \lVert f^{(4)} \rVert_\infty .
$$

For the piecewise linear spline, $\lVert f - s \rVert_\infty \le \frac{h^2}{8} \lVert f'' \rVert_\infty$.

**Theorem 9 (Minimum-curvature / minimum-energy property).** Among all $g \in C^2[a,b]$ with $g(x_i) = y_i$, the **natural** cubic spline $s$ uniquely minimizes the bending energy

$$
E[g] = \int_a^b \bigl( g''(x) \bigr)^2 dx .
$$

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — Existence and uniqueness of the interpolating polynomial

**Claim.** (Theorem 1.) Exactly one $p \in \mathbb{P}_n$ interpolates $n+1$ values at distinct nodes.

**Existence (constructive).** Define $L_i$ as in Definition 2. The denominator $\prod_{j \neq i}(x_i - x_j)$ is nonzero because the nodes are distinct, and $L_i(x_j) = \delta_{ij}$ by inspection: the numerator contains the factor $(x_j - x_j) = 0$ whenever $j \neq i$, and at $x = x_i$ numerator and denominator coincide. Hence $p(x) = \sum_i y_i L_i(x)$ lies in $\mathbb{P}_n$ and satisfies $p(x_j) = \sum_i y_i \delta_{ij} = y_j$.

**Uniqueness.** Suppose $p, q \in \mathbb{P}_n$ both interpolate. Then $d = p - q \in \mathbb{P}_n$ vanishes at the $n+1$ distinct points $x_0, \ldots, x_n$. A nonzero polynomial of degree $\le n$ has at most $n$ roots (fundamental theorem of algebra), so $d \equiv 0$ and $p = q$.

**Linear-algebra restatement.** In the monomial basis the conditions read $Va = y$ with $V_{ij} = x_i^{\,j}$. Uniqueness for all right-hand sides is equivalent to $\det V \neq 0$, and the classical evaluation

$$
\det V = \prod_{0 \le i \lt j \le n} (x_j - x_i)
$$

is proved by induction: subtracting $x_0$ times column $j-1$ from column $j$ (top to bottom) makes the first row $(1, 0, \ldots, 0)$ and factors $(x_i - x_0)$ out of row $i$, leaving a Vandermonde determinant of size $n$. $\blacksquare$

Note the warning already visible here: distinct nodes make $\det V \neq 0$, but for equispaced nodes $\det V$ is *exponentially small* relative to the matrix norm, so the monomial-basis approach is catastrophically ill-conditioned.

### Proof 2 — The interpolation error theorem

**Claim.** (Theorem 2.) For $f \in C^{n+1}[a,b]$ and $x$ fixed,

$$
f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!}\,\omega_{n+1}(x), \qquad \omega_{n+1}(x) = \prod_{i=0}^{n}(x - x_i).
$$

**Proof.** If $x$ equals some node both sides vanish, so assume $x \notin \{x_i\}$ and hence $\omega_{n+1}(x) \neq 0$. Define the constant

$$
K = \frac{f(x) - p_n(x)}{\omega_{n+1}(x)}
$$

and the auxiliary function of a new variable $t$:

$$
g(t) = f(t) - p_n(t) - K\,\omega_{n+1}(t).
$$

Then $g \in C^{n+1}[a,b]$ and $g$ vanishes at the $n+2$ distinct points $x_0, \ldots, x_n$ and $x$: at each node because $f(x_i) = p_n(x_i)$ and $\omega_{n+1}(x_i) = 0$; at $t = x$ by the definition of $K$.

Apply **Rolle's theorem** repeatedly. Between consecutive zeros of $g$ there is a zero of $g'$, so $g'$ has at least $n+1$ zeros; $g''$ has at least $n$; by induction $g^{(n+1)}$ has at least one zero, at some $\xi_x$ in the open interval spanned by $x$ and the nodes. Now differentiate $g$ exactly $n+1$ times: $p_n^{(n+1)} \equiv 0$ (degree $\le n$), and $\omega_{n+1}$ is monic of degree $n+1$ so $\omega_{n+1}^{(n+1)} \equiv (n+1)!$. Hence

$$
0 = g^{(n+1)}(\xi_x) = f^{(n+1)}(\xi_x) - K\,(n+1)! \implies K = \frac{f^{(n+1)}(\xi_x)}{(n+1)!}.
$$

Substituting $K$ back into its definition gives the claim. $\blacksquare$

**Reading the formula.** Two independent factors: $f^{(n+1)}$ is a property of the *function* (uncontrollable), while $\omega_{n+1}$ is a property of the *node placement* (fully controllable). Everything that follows — Runge, Chebyshev, splines — is an attempt to control the second factor when the first misbehaves.

### Proof 3 — Divided differences are normalized derivatives

**Claim.** (Theorem 3.) $f[x_0, \ldots, x_n]$ is the leading coefficient of $p_n$, and equals $f^{(n)}(\xi)/n!$ for some $\xi$ in the node span.

**Step 1 — leading coefficient.** Induct on $n$. Let $p_{0..n-1}$ interpolate at $x_0, \ldots, x_{n-1}$ and $p_{1..n}$ at $x_1, \ldots, x_n$. The combination

$$
p(x) = \frac{(x - x_0)\,p_{1..n}(x) - (x - x_n)\,p_{0..n-1}(x)}{x_n - x_0}
$$

is in $\mathbb{P}_n$ and reproduces the data at all of $x_0, \ldots, x_n$ (check the three cases $x = x_0$, $x = x_n$, $x = x_j$ interior), so by uniqueness $p = p_n$. Comparing leading coefficients gives exactly the recursion of Definition 3, and induction identifies $f[x_0,\ldots,x_n]$ with the leading coefficient of $p_n$.

**Step 2 — mean-value form.** Set $g = f - p_{n-1}$ where $p_{n-1}$ interpolates $f$ at $x_0, \ldots, x_{n-1}$. Also let $q_n$ interpolate at all $n+1$ nodes; then $q_n - p_{n-1}$ is a degree-$n$ polynomial vanishing at $x_0, \ldots, x_{n-1}$ with leading coefficient $f[x_0,\ldots,x_n]$. Alternatively and more directly: the function $E = f - p_n$ vanishes at $n+1$ points, so by the repeated-Rolle argument of Proof 2 applied $n$ times to $f - p_{n-1}$ (which has $n$ zeros), $f^{(n)} - p_{n-1}^{(n)}$... more cleanly, $f - p_n$ has $n+1$ zeros so $(f - p_n)^{(n)}$ has a zero $\xi$. Since $p_n^{(n)} \equiv n!\,f[x_0,\ldots,x_n]$ (constant, from Step 1),

$$
0 = f^{(n)}(\xi) - n!\, f[x_0, \ldots, x_n] \implies f[x_0, \ldots, x_n] = \frac{f^{(n)}(\xi)}{n!}. \qquad \blacksquare
$$

**Consequences.**
- Divided differences are **symmetric** in their arguments (they equal a leading coefficient, which does not care about ordering).
- Confluent limits define Hermite data: $f[x_0, x_0] = f'(x_0)$, and generally $f[\underbrace{x, \ldots, x}_{k+1}] = f^{(k)}(x)/k!$.
- Adding one node to the Newton form costs $O(n)$, not a full re-solve — the crucial advantage over the Lagrange form.
- The secant method's golden-ratio error recursion (Topic 02) is the divided-difference identity $f(x) - p_1(x) = f[x_0, x_1, x](x - x_0)(x - x_1)$ in disguise.

### Proof 4 — Chebyshev nodes minimize the node polynomial

**Claim.** (Theorem 4.) $\tilde T_n = 2^{1-n}T_n$ is the unique monic degree-$n$ polynomial minimizing $\lVert q \rVert_{\infty,[-1,1]}$, with minimum value $2^{1-n}$.

**Step 1 — $\tilde T_n$ is monic with sup norm $2^{1-n}$.** From $T_{n+1}(x) = 2x T_n(x) - T_{n-1}(x)$ with $T_0 = 1$, $T_1 = x$, the leading coefficient of $T_n$ is $2^{n-1}$ for $n \ge 1$; dividing gives a monic polynomial. Since $T_n(\cos\theta) = \cos n\theta$, we have $\lvert T_n \rvert \le 1$ on $[-1,1]$ with equioscillation: at $y_k = \cos(k\pi/n)$, $k = 0,\ldots,n$, $T_n(y_k) = (-1)^k$. Hence $\lVert \tilde T_n \rVert_\infty = 2^{1-n}$, attained with alternating signs at $n+1$ points.

**Step 2 — optimality by contradiction.** Suppose some monic $q \in \mathbb{P}_n$ has $\lVert q \rVert_\infty \lt 2^{1-n}$. Consider $d = \tilde T_n - q$. Because both are monic of degree $n$, $d \in \mathbb{P}_{n-1}$. At the $n+1$ alternation points,

$$
d(y_k) = (-1)^k 2^{1-n} - q(y_k),
$$

and since $\lvert q(y_k) \rvert \lt 2^{1-n}$, the sign of $d(y_k)$ is the sign of $(-1)^k$. So $d$ alternates in sign at $n+1$ consecutive points, hence has at least $n$ distinct roots by the intermediate value theorem. But $d \in \mathbb{P}_{n-1}$ with $n$ roots forces $d \equiv 0$, i.e. $q = \tilde T_n$, contradicting $\lVert q \rVert_\infty \lt 2^{1-n} = \lVert \tilde T_n \rVert_\infty$. $\blacksquare$

**Consequence for interpolation.** $\omega_{n+1}$ is monic of degree $n+1$; taking its roots (= the nodes) to be the roots of $T_{n+1}$ makes $\omega_{n+1} = \tilde T_{n+1}$ and $\lVert \omega_{n+1} \rVert_\infty = 2^{-n}$. Then

$$
\lVert f - p_n \rVert_\infty \le \frac{\lVert f^{(n+1)} \rVert_\infty}{2^{n} (n+1)!},
$$

versus the equispaced bound whose $\lVert \omega_{n+1} \rVert_\infty \approx n!\,h^{n+1}/e$ is larger by a factor that grows exponentially in $n$. For $n = 20$ on $[-1,1]$ the Chebyshev node polynomial is smaller by more than four orders of magnitude.

### Proof 5 — The Runge phenomenon, quantified

**Setup.** Let $f(x) = \frac{1}{1 + 25 x^2}$ on $[-1,1]$ and let $p_n$ interpolate at $n+1$ **equispaced** nodes. Then $\lVert f - p_n \rVert_\infty \to \infty$ as $n \to \infty$, and the divergence is geometric.

**Why the error theorem does not save us.** $f$ is $C^\infty$ on $[-1,1]$, but it has poles at $x = \pm i/5$ in the complex plane. Cauchy's estimate gives $\lvert f^{(n+1)}(x) \rvert \sim (n+1)!\, 5^{\,n+1}$ — the derivatives grow *faster* than $(n+1)!$, so the $1/(n+1)!$ in the error theorem is completely overwhelmed. Whether the interpolant converges is then decided entirely by $\omega_{n+1}$.

**Why $\omega_{n+1}$ is lopsided for equispaced nodes.** With $h = 2/n$ and nodes $x_i = -1 + ih$, take the two evaluation points $x = 0$ (centre, $n$ even) and $x = 1 - h/2$ (near the end). Writing $\lvert \omega_{n+1}(x) \rvert = \prod_i \lvert x - x_i \rvert$ and applying $\log$, the centre point sees distances $h/2, 3h/2, \ldots$ symmetric on both sides, while the endpoint sees distances $h/2, 3h/2, \ldots$ all on one side. The one-sided product is essentially $h^{n+1}\,\frac{(n+1)!}{2^{n+1}} \cdot \frac{1}{\sqrt{\pi n}}$-ish, while the two-sided product involves the much smaller $\bigl((n/2)!\bigr)^2$. The upshot is the classical ratio

$$
\frac{\max_{x} \lvert \omega_{n+1}(x) \rvert}{\lvert \omega_{n+1}(0) \rvert} \sim 2^{\,n},
$$

so $\omega_{n+1}$ is exponentially larger near the ends than in the middle.

**Potential-theoretic statement (the sharp result).** For equispaced nodes the interpolant of $f$ converges only where

$$
\lvert x \rvert \lt 0.7266\ldots \quad \text{(for } f = 1/(1+25x^2)\text{)},
$$

and diverges geometrically outside; the threshold is where the equilibrium potential of the uniform measure equals the distance to the singularity. The general principle (Runge's theorem, made precise by Walsh and by Trefethen): equispaced interpolation converges only if $f$ is analytic in a specific lens-shaped region far larger than the Chebyshev "Bernstein ellipse."

**The Chebyshev cure.** With Chebyshev nodes, $\omega_{n+1}$ is *equioscillating* — the same size everywhere — and convergence for $f$ analytic in a Bernstein ellipse of parameter $\rho \gt 1$ is geometric:

$$
\lVert f - p_n \rVert_\infty = O(\rho^{-n}).
$$

For $f = 1/(1+25x^2)$, $\rho = 5 + \sqrt{26} \approx 10.1$, so the Chebyshev interpolant converges at roughly one digit per node. $\blacksquare$

**Moral.** Runge's phenomenon is not caused by rounding, by high degree per se, or by non-smoothness. It is caused by *equispaced sampling*, and it is cured by clustering nodes at the endpoints — or by refusing to raise the degree at all.

### Proof 6 — The cubic spline tridiagonal system

**Setup.** Knots $x_0 \lt \cdots \lt x_n$, $h_i = x_{i+1} - x_i$, unknown moments $M_i = s''(x_i)$.

**Step 1 — $s''$ is piecewise linear.** On $[x_i, x_{i+1}]$ the spline is a cubic, so $s''$ is linear and determined by its endpoint values:

$$
s''(x) = M_i \frac{x_{i+1} - x}{h_i} + M_{i+1} \frac{x - x_i}{h_i}.
$$

Continuity of $s''$ at the knots is automatic in this parameterisation — that is the whole point of using moments as unknowns.

**Step 2 — integrate twice and impose interpolation.** Integrating twice introduces two constants per interval, fixed by $s(x_i) = y_i$ and $s(x_{i+1}) = y_{i+1}$:

$$
s(x) = M_i \frac{(x_{i+1} - x)^3}{6 h_i} + M_{i+1} \frac{(x - x_i)^3}{6 h_i} + \left( \frac{y_i}{h_i} - \frac{M_i h_i}{6} \right)(x_{i+1} - x) + \left( \frac{y_{i+1}}{h_i} - \frac{M_{i+1} h_i}{6} \right)(x - x_i).
$$

**Step 3 — impose continuity of $s'$.** Differentiating and evaluating at the right end of $[x_{i-1}, x_i]$ and the left end of $[x_i, x_{i+1}]$:

$$
s'(x_i^-) = \frac{h_{i-1}}{6}\,M_{i-1} + \frac{h_{i-1}}{3}\,M_i + \frac{y_i - y_{i-1}}{h_{i-1}}, \qquad
s'(x_i^+) = -\frac{h_i}{3}\,M_i - \frac{h_i}{6}\,M_{i+1} + \frac{y_{i+1} - y_i}{h_i}.
$$

Setting $s'(x_i^-) = s'(x_i^+)$ and multiplying by $6$:

$$
h_{i-1} M_{i-1} + 2(h_{i-1} + h_i) M_i + h_i M_{i+1} = 6\left( \frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}} \right), \quad i = 1, \ldots, n-1 .
$$

**Step 4 — close the system.** Natural conditions set $M_0 = M_n = 0$, leaving an $(n-1) \times (n-1)$ system. Clamped conditions add two rows involving $s'(x_0) = f'(a)$ and $s'(x_n) = f'(b)$. In all cases the matrix is tridiagonal with $\lvert 2(h_{i-1} + h_i) \rvert \gt h_{i-1} + h_i$, i.e. **strictly diagonally dominant**, hence nonsingular, and the Thomas algorithm (LU without pivoting, provably stable here) solves it in $O(n)$ time and $O(n)$ memory. $\blacksquare$

Note the structure: the right-hand side is $6\bigl(f[x_i, x_{i+1}] - f[x_{i-1}, x_i]\bigr)$, i.e. $12 \cdot$ a second divided difference — a discrete second derivative, exactly what one expects to determine second-derivative unknowns.

### Proof 7 — The natural spline minimizes bending energy

**Claim.** (Theorem 9.) Let $s$ be the natural cubic spline interpolating $(x_i, y_i)$ and let $g \in C^2[a,b]$ be any other interpolant of the same data. Then $\int_a^b (g'')^2 \ge \int_a^b (s'')^2$, with equality iff $g = s$.

**Proof.** Write $\eta = g - s$, so $\eta(x_i) = 0$ for all $i$. Expand:

$$
\int_a^b (g'')^2 = \int_a^b (s'')^2 + 2\int_a^b s'' \eta'' + \int_a^b (\eta'')^2 .
$$

It suffices to show the cross term vanishes. Integrate by parts:

$$
\int_a^b s'' \eta'' = \bigl[ s'' \eta' \bigr]_a^b - \int_a^b s''' \eta' .
$$

The boundary term is zero because the **natural** conditions give $s''(a) = s''(b) = 0$. For the remaining integral, split over the knot intervals; on each $[x_i, x_{i+1}]$ the spline is cubic so $s''' \equiv c_i$ is constant:

$$
\int_a^b s''' \eta' = \sum_{i=0}^{n-1} c_i \int_{x_i}^{x_{i+1}} \eta' = \sum_{i=0}^{n-1} c_i \bigl[ \eta(x_{i+1}) - \eta(x_i) \bigr] = 0,
$$

since $\eta$ vanishes at every knot. Therefore

$$
\int_a^b (g'')^2 = \int_a^b (s'')^2 + \int_a^b (\eta'')^2 \ge \int_a^b (s'')^2 ,
$$

with equality iff $\eta'' \equiv 0$, i.e. $\eta$ is affine; an affine function vanishing at $n+1 \ge 2$ points is identically zero, so $g = s$. $\blacksquare$

**Interpretation.** $\int (s'')^2$ is the linearized bending energy of an elastic beam (a draughtsman's "spline") constrained to pass through pins at the data points — the physical origin of the name. The same functional, penalized rather than constrained, defines the **smoothing spline**: minimize $\sum_i (y_i - g(x_i))^2 + \lambda \int (g'')^2$, whose solution is a natural cubic spline with knots at the data — the workhorse of generalized additive models.

## 4. Computational & Algorithmic Insights

### Choosing a representation

| Form | Setup cost | Cost per evaluation | Add a node | Stability |
| :--- | :--- | :--- | :--- | :--- |
| Monomial (solve Vandermonde) | $O(n^3)$ | $O(n)$ Horner | re-solve | very poor (exponential conditioning) |
| Lagrange (classical) | none | $O(n^2)$ | $O(n^2)$ | fine but slow |
| Newton divided differences | $O(n^2)$ | $O(n)$ | $O(n)$ | good with ordered nodes |
| Barycentric Lagrange | $O(n^2)$ (weights) | $O(n)$ | $O(n)$ | backward stable |

The **barycentric formula** is the modern default:

$$
p_n(x) = \frac{\displaystyle\sum_{i=0}^{n} \frac{w_i}{x - x_i} y_i}{\displaystyle\sum_{i=0}^{n} \frac{w_i}{x - x_i}}, \qquad w_i = \frac{1}{\prod_{j \neq i}(x_i - x_j)} .
$$

For Chebyshev points of the second kind the weights collapse to $w_i = (-1)^i \delta_i$ with $\delta_i = \tfrac{1}{2}$ at the ends and $1$ otherwise — no computation at all, and evaluation in $O(n)$ flops. This is what Chebfun does, and it is stable for degrees in the thousands.

```python
import numpy as np

def bary_cheb(f, n, xq):
    # Barycentric interpolation at n+1 Chebyshev points of the 2nd kind on [-1, 1]
    k = np.arange(n + 1)
    x = np.cos(np.pi * k / n)
    y = f(x)
    w = (-1.0) ** k
    w[0] *= 0.5
    w[-1] *= 0.5
    xq = np.atleast_1d(np.asarray(xq, float))
    num = np.zeros_like(xq)
    den = np.zeros_like(xq)
    out = np.full_like(xq, np.nan)
    exact = np.zeros(xq.shape, dtype=bool)
    for xi, yi, wi in zip(x, y, w):
        d = xq - xi
        hit = d == 0.0
        out[hit] = yi
        exact |= hit
        d = np.where(hit, 1.0, d)
        num += wi * yi / d
        den += wi / d
    return np.where(exact, out, num / den)
```

### Conditioning of the Vandermonde matrix

Solving $Va = y$ for monomial coefficients is a textbook example of an avoidable disaster. For $n+1$ equispaced nodes on $[0,1]$ the condition number grows roughly as

$$
\kappa_2(V) \sim \frac{(1 + \sqrt{2})^{\,n+1}}{\sqrt{\pi n}} \approx 2.4^{\,n},
$$

so $n = 20$ already costs about 8 significant digits and $n = 40$ loses everything in double precision. Two independent fixes:

1. **Change the basis.** Chebyshev or Newton bases give condition numbers that are polynomial rather than exponential in $n$; the barycentric form avoids forming any coefficient vector at all.
2. **Change the nodes.** Even in the monomial basis, Chebyshev nodes reduce $\kappa_2(V)$ dramatically — though the basis change matters more.

The rule of practice: **never** compute interpolating-polynomial coefficients by solving a Vandermonde system. `numpy.polyfit` at high degree emits an ill-conditioning warning for exactly this reason; use `numpy.polynomial.chebyshev` or a barycentric evaluator instead.

### Piecewise methods and the cost ledger

Splines replace one hard global problem with many easy local ones:

| Method | Smoothness | Global error | Setup cost | Locality of a data change |
| :--- | :--- | :--- | :--- | :--- |
| Piecewise constant | none | $O(h)$ | $O(n)$ | 1 interval |
| Piecewise linear | $C^0$ | $O(h^2)$, const $1/8$ | $O(n)$ | 2 intervals |
| Cubic Hermite (PCHIP) | $C^1$ | $O(h^4)$ | $O(n)$ | 2 intervals |
| Cubic spline | $C^2$ | $O(h^4)$, const $5/384$ | $O(n)$ tridiagonal | global, but decaying geometrically |
| Global polynomial (Chebyshev) | $C^\infty$ | geometric for analytic $f$ | $O(n \log n)$ | global |

Two practical remarks. First, cubic-spline coefficients are *not* local — changing one $y_i$ perturbs the whole curve — but the perturbation decays like $(2 - \sqrt3)^{\,\lvert j - i \rvert} \approx 0.268^{\,\lvert j - i \rvert}$, so it is local for all practical purposes. Second, splines do **not** preserve monotonicity or convexity; if the data are monotone and you need the interpolant to be too (probability calibration curves, cumulative dose–response), use **PCHIP** (`scipy.interpolate.PchipInterpolator`), which sacrifices $C^2$ and one order of accuracy for a shape-preserving guarantee.

**B-splines glimpse.** The basis $\{B_{i,k}\}$ defined by the Cox–de Boor recursion

$$
B_{i,0}(x) = \mathbf{1}_{[t_i, t_{i+1})}(x), \qquad B_{i,k}(x) = \frac{x - t_i}{t_{i+k} - t_i} B_{i,k-1}(x) + \frac{t_{i+k+1} - x}{t_{i+k+1} - t_{i+1}} B_{i+1,k-1}(x)
$$

spans the same spline space but with **local support** ($B_{i,k}$ is nonzero only on $k+1$ intervals), non-negativity, and a partition of unity $\sum_i B_{i,k} \equiv 1$. Consequently the collocation matrix is banded and totally positive, the curve lies in the convex hull of its control points, and evaluation via de Boor's algorithm is numerically stable. This is the representation used in CAD, in `scipy.interpolate.BSpline`, and in the spline layers of Kolmogorov–Arnold networks.

### Practical algorithm summary

```python
import numpy as np
from scipy.interpolate import CubicSpline, PchipInterpolator, BarycentricInterpolator

x = np.linspace(-1, 1, 15)
f = lambda t: 1.0 / (1.0 + 25.0 * t**2)

# 1. Global polynomial at equispaced nodes -- DIVERGES (Runge)
p_equi = BarycentricInterpolator(x, f(x))

# 2. Global polynomial at Chebyshev nodes -- converges geometrically
xc = np.cos(np.pi * np.arange(15) / 14)
p_cheb = BarycentricInterpolator(xc, f(xc))

# 3. Cubic spline, not-a-knot (SciPy default) -- O(h^4), no blow-up
s = CubicSpline(x, f(x))                    # bc_type='not-a-knot'
s_nat = CubicSpline(x, f(x), bc_type='natural')
s_cl = CubicSpline(x, f(x), bc_type=((1, 0.0), (1, 0.0)))   # clamped

# 4. Shape-preserving monotone interpolation
mono = PchipInterpolator(np.sort(x), np.sort(f(x)))
```

Checklist for choosing a method:

- Data are **exact** samples of a smooth function on a finite interval, and you may choose where to sample $\Rightarrow$ Chebyshev nodes + barycentric evaluation.
- Data are exact but the nodes are **given** and equispaced $\Rightarrow$ cubic spline (not-a-knot), never a high-degree global polynomial.
- Data are **noisy** $\Rightarrow$ do not interpolate; use a smoothing spline or regression (penalize $\int (g'')^2$).
- Data must stay **monotone or positive** $\Rightarrow$ PCHIP or a constrained/monotone spline.
- You need derivatives too $\Rightarrow$ Hermite / cubic-spline derivative, remembering that differentiating an interpolant loses one order of accuracy.

## 5. Real-World Physics & AI/ML Applications

**Ephemerides and tabulated physics.** JPL planetary ephemerides are distributed as **Chebyshev coefficients** over short time blocks, not as raw positions: a client reconstructs a planet's position to millimetre accuracy by evaluating a degree-12 Chebyshev series. Steam tables, equation-of-state tables, and opacity tables in astrophysics are likewise interpolated — usually bicubic splines — inside tight simulation loops where each evaluation must cost microseconds.

**Computer graphics and CAD.** Every font glyph is a piecewise Bézier curve (Bernstein-basis polynomials, a close relative of B-splines); every industrial surface is a NURBS patch. The convex-hull and variation-diminishing properties of the B-spline basis are what make interactive control-point editing predictable.

**Signal reconstruction.** Shannon interpolation (sinc kernels) is the band-limited analogue of polynomial interpolation; in practice image resampling uses cubic convolution ($C^1$ piecewise cubics, the Catmull–Rom / Keys kernel) because it is local, fast, and $O(h^3)$ accurate — the resize operation in every deep-learning image pipeline is a spline evaluation.

**Numerical methods built on interpolation.** Nearly every algorithm downstream is an interpolant integrated, differentiated, or root-found:
- Quadrature rules (Topic 06) are $\int$ of the interpolant; Newton–Cotes weights are $\int L_i$.
- Finite-difference stencils (Topic 05) are $\frac{d}{dx}$ of the interpolant.
- The secant and Muller root-finders (Topic 02) invert an interpolant.
- Multistep ODE solvers (Adams–Bashforth/Moulton) integrate an interpolant of past derivative values.

**Machine learning applications.**

- **Learning-rate schedules.** Cosine annealing is literally a smooth interpolant between endpoint learning rates; warmup-then-decay schedules in modern LLM training are specified at milestones and interpolated (linear or cosine) in between. Piecewise-linear schedules are degree-1 splines; the "one-cycle" policy is a two-piece cosine spline.

- **Spline layers and KANs.** Kolmogorov–Arnold Networks replace fixed activations with learnable univariate B-spline functions on each edge, $\phi(x) = \sum_i c_i B_{i,3}(x)$, with the coefficients $c_i$ as parameters. Local support means each input activates only $k+1$ basis functions, which gives sparse gradients and a natural grid-refinement scheme (knot insertion) for progressive capacity growth. The same idea appears in **deep spline / linear-spline activation** networks and in normalizing flows built from monotone rational-quadratic splines (Neural Spline Flows), where monotonicity of the spline is what guarantees invertibility and a triangular Jacobian.

- **Probability calibration.** Reliability diagrams map predicted confidence to observed accuracy; **isotonic regression** and **spline calibration** fit a monotone interpolant/regressor to binned data. Monotonicity is non-negotiable here — an unconstrained cubic spline can produce a non-monotone (hence nonsensical) calibration map, which is why PCHIP or isotonic fits are used.

- **Hyperparameter and neural-architecture search.** Response-surface methods interpolate a small number of expensive evaluations to propose new points; Gaussian-process regression is the statistical generalization (kernel interpolation), and the classical spline connection is exact — the natural cubic spline is the posterior mean of a GP with a twice-integrated Brownian-motion prior.

- **Resampling and augmentation.** Time-series augmentation (time warping, resampling to a fixed length), audio resampling, and coordinate-grid sampling (`torch.nn.functional.grid_sample` with `mode='bicubic'`) all rest on cubic interpolation. Positional-embedding interpolation when fine-tuning a Vision Transformer at a new resolution is bicubic spline interpolation of a learned grid.

- **Distillation of expensive functions.** Softmax temperature curves, tokenizer cost models, and lookup-table approximations of transcendental activations (GELU tables on edge accelerators) are all piecewise-polynomial interpolants chosen to hit a target sup-norm error with the fewest table entries.

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Existence/uniqueness | Unique $p \in \mathbb{P}_n$ through $n+1$ distinct nodes; $\det V = \prod_{i \lt j}(x_j - x_i)$ |
| Error theorem | $f(x) - p_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\,\omega_{n+1}(x)$ |
| Divided differences | $f[x_0,\ldots,x_n] = \frac{f^{(n)}(\xi)}{n!}$ = leading coefficient of $p_n$ |
| Chebyshev minimax | $\min$ over monic $q \in \mathbb{P}_n$ of $\Vert q \Vert_\infty = 2^{1-n}$ on $[-1,1]$ |
| Lebesgue constants | equispaced $\sim 2^{n+1}/(e\,n\log n)$; Chebyshev $\sim \frac{2}{\pi}\log n$ |
| Runge phenomenon | equispaced interpolation of $1/(1+25x^2)$ diverges geometrically |
| Hermite error | $\frac{f^{(2n+2)}(\xi)}{(2n+2)!}\prod (x - x_i)^2$ |
| Cubic spline system | tridiagonal, diagonally dominant, $O(n)$ via Thomas |
| Spline accuracy | clamped: $\Vert f - s \Vert_\infty \le \frac{5}{384} h^4 \Vert f^{(4)} \Vert_\infty$ |
| Minimum energy | natural spline uniquely minimizes $\int (g'')^2$ over $C^2$ interpolants |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Lagrange/Newton forms, divided differences | Burden & Faires, *Numerical Analysis* | Ch. 3.1–3.2 |
| Interpolation error theorem | Burden & Faires; Quarteroni et al. | Ch. 3.1; Ch. 8.1 |
| Hermite interpolation | Burden & Faires | Ch. 3.4 |
| Cubic splines and end conditions | Burden & Faires; de Boor | Ch. 3.5; *A Practical Guide to Splines* Ch. 4 |
| B-splines, Cox–de Boor, knot insertion | de Boor | Chs. 9–11 |
| Barycentric formula and its stability | Berrut & Trefethen, SIAM Review 46 (2004) | entire paper |
| Chebyshev minimax, Lebesgue constants | Trefethen, *ATAP* | Chs. 12–15 |
| Runge phenomenon, potential theory | Trefethen, *ATAP* | Ch. 13; Ch. 18 |
| Vandermonde conditioning | Trefethen & Bau, *Numerical Linear Algebra* | Lectures 12–13 |
| Minimum-energy property, smoothing splines | Green & Silverman | Chs. 2–3 |
| Practical caveats, code | Press et al., *Numerical Recipes* | Ch. 3 |

**Primary references.** Burden & Faires (Ch. 3); Trefethen, *Approximation Theory and Approximation Practice* (Chs. 5, 11–15); de Boor, *A Practical Guide to Splines*; Berrut & Trefethen (2004); Quarteroni, Sacco & Saleri (Ch. 8); Green & Silverman (1994); Press et al. (Ch. 3).